# NILM Model Training on Google Colab

Train a Non-Intrusive Load Monitoring (NILM) disaggregation model using the
`underscore-edge-model-pipeline` package. This notebook covers the full workflow: training,
evaluation, ONNX export, and downloading artefacts.

**Runtime:** Select *Runtime > Change runtime type > T4 GPU* before running.

## 1. Install

The repo is private, so clone it with a [personal access token](https://github.com/settings/tokens) stored in Colab Secrets as `GITHUB_TOKEN`.

In [1]:
import os
IS_HOSTED_COLAB = "COLAB_RELEASE_TAG" in os.environ
print("Hosted Colab" if IS_HOSTED_COLAB else "Local Colab runtime")

Hosted Colab


In [2]:
if IS_HOSTED_COLAB:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
else:
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN via: docker run -e GITHUB_TOKEN=ghp_... ..."
        )

!git clone https://{token}@github.com/underscoreHQ/underscore-edge-model-pipeline.git
%pip install underscore-edge-model-pipeline/

Cloning into 'underscore-edge-model-pipeline'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 304 (delta 157), reused 252 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (304/304), 392.37 KiB | 8.01 MiB/s, done.
Resolving deltas: 100% (157/157), done.
Processing ./underscore-edge-model-pipeline
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB

## 2. GPU check

In [3]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — training will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

No GPU detected — training will be slow.
Go to Runtime > Change runtime type > T4 GPU


## 3. Train

In [ ]:
import sys
sys.path.append('/content/underscore-edge-model-pipeline/src')

from edge_pipeline.training.trainer import train_model

model, trainer = train_model(
    model_type="atcn",
    window_size=288,
    batch_size=128,
    max_epochs=30,
    stride=5,
    learning_rate=1e-3,
    use_amda=True,
    amda_scale=2.5,
    num_workers=2,
    output_dir="outputs",
    use_wandb=True,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zainelabedeen (zainelabedeen-underscore-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss_fn      │ MSELoss            │      0 │ train │     0 │
│ 1 │ input_proj   │ Conv1d             │    256 │ train │     0 │
│ 2 │ tcn          │ Sequential         │  790 K │ train │     0 │
│ 3 │ pos_encoding │ PositionalEncoding │      0 │ train │     0 │
│ 4 │ transformer  │ TransformerEncoder │  396 K │ train │     0 │
│ 5 │ norm         │ LayerNorm          │    256 │ train │     0 │
│ 6 │ fc           │ Sequential         │ 17.2 K │ train │     0 │
└───┴──────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 1.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.2 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 97                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is 
set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

## 4. Evaluate

In [ ]:
from edge_pipeline.data.datamodule import NilmDataModule
from edge_pipeline.evaluation.evaluate import APPLIANCE_NAMES, evaluate_model

# Set up a datamodule for evaluation (no augmentation)
eval_dm = NilmDataModule(
    window_size=288,
    batch_size=128,
    num_workers=2,
    use_amda=False,
)
eval_dm.setup()

device = "cuda" if torch.cuda.is_available() else "cpu"
results = evaluate_model(model, eval_dm, device=device)

print(f"{'Metric':<12} {'Value':>10}")
print("-" * 24)
for key in ["mae", "mse", "rmse", "r2", "nde"]:
    print(f"{key:<12} {results[key]:>10.4f}")

print(f"\n{'Appliance':<14} {'MAE':>10} {'R2':>10}")
print("-" * 36)
for name in APPLIANCE_NAMES:
    print(f"{name:<14} {results[f'{name}_mae']:>10.4f} {results[f'{name}_r2']:>10.4f}")

## 5. Export to ONNX

In [ ]:
from edge_pipeline.export.onnx_exporter import OnnxExporter

exporter = OnnxExporter(model, output_dir="exports", model_name="nilm_colab")
normalisation_metadata = eval_dm.get_normalisation_metadata()
onnx_path = exporter.export(normalisation_metadata=normalisation_metadata)
print(f"Exported ONNX model to {onnx_path}")

## 6. Download artefacts

In [ ]:
if IS_HOSTED_COLAB:
    from google.colab import files
    files.download(str(onnx_path))
    files.download("exports/nilm_colab_metadata.json")
else:
    print(f"Artefacts saved to:\n  {onnx_path}\n  exports/nilm_colab_metadata.json")
    print("Copy from the container with: docker cp <container>:/content/exports/ ./exports/")